In [1]:
import matplotlib as mpl

mpl.use("Agg")
import re
import os
import sys
import glob
import warnings
import gc
import xarray as xr
import cmocean as cm
from subprocess import Popen, PIPE, STDOUT
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm

    tqdm_avail = True
except:
    warnings.warn(
        "Optional dependency `tqdm` not found. This will make progressbars a lot nicer. \
    Install with `conda install -c conda-forge tqdm`"
    )
    tqdm_avail = False

In [2]:
dataset_name = "CM4"  # "OM4", "CM4"
prefix = dataset_name
# Long OM4 Rollout
# assert dataset_name == "OM4"
# pred_path = '/pscratch/sd/s/suryad/Ocean_Emulator/Preds/2024-09-12_ConvNextUNetTrain3Dv021Eval3DhfdsanomsNetZeroHfTempOnly1975Epochs70Epoch55Years100_10repeat_36_6k_Train_global_3D_Test_global_3D_all_N_train_0_Lateral_Data_025_no_smooth/Pred_lateral_Fast_Data_025_global_3D_all_N_samples_0_rand_seed_1.zarr'
# long_roll = True

# Short OM4 Rollout
# assert dataset_name == "OM4"
# pred_path = "/pscratch/sd/s/suryad/Ocean_Emulator/Preds/2024-09-20_ConvNextUNetTrain3Dv021Eval3Dhfdsanoms1975NofastinoutEpochs70Epoch55_Train_global_3D_Test_global_3D_all_N_train_2850_Lateral_Data_025_no_smooth/Pred_lateral_Fast_Data_025_global_3D_all_N_samples_2850_rand_seed_1.zarr"
# long_roll = False

# Short CM4 Rollout
assert dataset_name == "CM4"
pred_path = "/pscratch/sd/s/suryad/Ocean_Emulator/Preds/2024-11-22_ConvNextUNetCM4Hist1NofastinoutEpochs70Epoch55_Train_global_3D_Test_global_3D_all_N_train_13800_Lateral_Data_025_no_smooth/Pred_lateral_Fast_Data_025_global_3D_all_N_samples_13800_rand_seed_1.zarr"
long_roll = False

levels = 19

In [3]:
var_list = {
    "vo": r"$v$ $( m/s )$",
    "uo": r"$u$ $( m/s )$",
    "thetao": r"$T$ $( ^\circ C )$",
    "so": r"$so$ $( psu )$",
    "zos": r"$zos$ $( m )$",
}

In [4]:
def post_processor(ds: xr.Dataset, ds_truth: xr.Dataset, ls) -> xr.Dataset:
    """Converts the prediction output to an xarray dataset with the same dimensions/variables as input"""

    # correct swapped dimensions and warn
    if len(ds.x) == 180 and len(ds.y) == 360:
        ds = ds.rename({"x": "x_i", "y": "y_i"}).rename({"x_i": "y", "y_i": "x"})
        warnings.warn(
            "Swapped x and y dimensions detected. Fixing this now, but should be corrected upstream"
        )
    key = list(ds.variables.keys())[0]
    da = ds[key]
    n_lev = 19
    if set(ls) - {"zos"} == set(["uo", "vo", "thetao", "so"]):
        variables = ["uo", "vo", "thetao", "so"]
    elif set(ls) - {"zos"} == set(["thetao", "so"]):
        variables = ["thetao", "so"]
    elif set(ls) - {"zos"} == set(["uo", "vo"]):
        variables = ["uo", "vo"]
    elif set(ls) - {"zos"} == set(["thetao"]):
        variables = ["thetao"]
    slices = [slice(i, i + n_lev) for i in range(0, len(variables) * n_lev, n_lev)]
    var_slices = {k: sl for k, sl in zip(variables, slices)}
    variables = {
        k: da.isel(var=sl).rename({"var": "lev"}) for k, sl in var_slices.items()
    }
    if "zos" in ls:
        variables["zos"] = da.isel(var=-1).squeeze()

    ds_out = xr.Dataset(variables)
    for var in ds_out.data_vars:
        if "lev" in ds_out[var].dims:
            ds_out[var] = ds_out[var].where(ds_truth.wetmask)
        else:
            ds_out[var] = ds_out[var].where(ds_truth.wetmask.isel(lev=0))

    ## attach all coordinates from input
    ds_out = ds_out.assign_coords({co: ds_truth[co] for co in ds_truth.coords})
    ds_out.attrs = ds.attrs

    return ds_out

In [5]:
def get_om4_gt(long_roll):
    ds_input = xr.open_zarr(
        os.path.join("/pscratch/sd/s/suryad/data", "OM4_5daily_v0.2.1.zarr")
    )
    if long_roll:
        time = 7300
        repeats = 40
        start_year = 1990
        years = 400

        ds_groundtruth = ds_groundtruth.sel(time=slice("1990-01-01", "1999-12-31"))
        ds_groundtruth = xr.concat([ds_groundtruth] * repeats, dim="time")
        dates = np.array(range(3, 365 * years, 5))
        base = DatetimeProlepticGregorian(start_year, 1, 1)
        all_dates = [base + timedelta(days=int(day - 1)) for day in dates]
        ds_groundtruth = ds_groundtruth.assign_coords(
            time=all_dates[: ds_groundtruth.time.size]
        )
        ds_groundtruth = ds_groundtruth.isel(time=slice(0, time))
    else:
        ds_input = ds_input.sel(time=slice("1975-01-01", None))
        ds_groundtruth = ds_input.isel(time=slice(2903, 2903 + 600)).isel(
            lev=slice(None, levels)
        )
    return ds_groundtruth


def get_cm4_gt():
    import fsspec

    fs_osn = fsspec.filesystem(
        "s3",
        profile="ocean_emulator",  ## This is the profile name you configured above.
    )
    mapper = fs_osn.get_mapper(
        "emulators/jbusecke/ocean-emulators/CM4_5daily_v0.4.0.zarr"
    )
    ds_input = xr.open_zarr(mapper, consolidated=True)
    ds_input = ds_input.drop_vars(["lat_b", "lon_b"])

    ds_groundtruth = ds_input.isel(time=slice(13941, 13941 + 600)).isel(
        lev=slice(None, levels)
    )
    return ds_groundtruth

In [6]:
if dataset_name == "OM4":
    ds_groundtruth = get_om4_gt(long_roll)
elif dataset_name == "CM4":
    ds_groundtruth = get_cm4_gt()
else:
    raise ValueError("Incorrect dataset name")

# ds_groundtruth = ds_groundtruth.astype("float32")

ds = xr.open_dataset(pred_path, engine="zarr", chunks={})
if long_roll:
    ds = ds.isel(time=slice(0, time))

ds_prediction = post_processor(
    ds, ds_groundtruth, ["thetao", "so", "zos"]
)  # Change in the future

/tmp/ipykernel_238153/1153242137.py:7: UserWarning: Swapped x and y dimensions detected. Fixing this now, but should be corrected upstream
  warnings.warn(


In [7]:
import numpy as np
from cartopy.mpl import geoaxes
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import dask.array as dsa


import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


# plotting basics
def _core_plot(ax, data, plotmethod=None, **kwargs):
    """Core plotting functionality"""

    # Deactivate cbar for contours (not sure this should be hardcoded...)
    if plotmethod == "contour":
        kwargs.pop("cbar_kwargs", None)

    # I am probably recoding something from matplotlib...is there a way to get
    # the plot.something functionslity with a keyword?
    # For now do it the hard way
    if plotmethod is None:
        p = data.plot(ax=ax, **kwargs)
    # doesnt work,...i want this for smoother images
    elif plotmethod == "imshow":
        # p = data.plot.imshow(ax=ax, **kwargs)
        # testing interpolation
        p = data.plot.imshow(ax=ax, interpolation="gaussian", **kwargs)
        # print(p.get_interpolation())
    elif plotmethod == "pcolormesh":
        p = data.plot.pcolormesh(ax=ax, **kwargs)
    elif plotmethod == "contour":
        p = data.plot.contour(ax=ax, **kwargs)
    elif plotmethod == "contourf":
        p = data.plot.contourf(ax=ax, **kwargs)
    else:
        raise RuntimeError(
            "Input '%s' not recognized \
        as plotmode"
            % plotmethod
        )
    return p


def _base_plot(
    ax, base_data, timestamp, timestep_value, framedim, plotmethod=None, **kwargs
):
    data = base_data.isel({framedim: timestamp})
    p = _core_plot(ax, data, plotmethod=plotmethod, **kwargs)
    return p


# Styling of the plot elements


def _style_dict_raw():
    return {
        "standard": {
            "bgcolor": "1.0",
            "fgcolor": "0.7",
            "landcolor": "0.2",
            "coastcolor": "0.3",
        },
        "dark": {
            "bgcolor": "0.1",
            "fgcolor": "0.7",
            "landcolor": "0.2",
            "coastcolor": "0.3",
        },
    }


def _style_dict(style=None):
    # set default style
    if style is None:
        style = "standard"
    # define parameters for styles
    style_dict = _style_dict_raw()
    return style_dict[style]


def _set_style(fig, ax, pp, style):
    "Sets the colorscheme for figure, axis and plot object (`pp`) according to style"
    # check if ax is 'normal' or cartopy projection
    is_geoax = False
    if isinstance(ax, geoaxes.GeoAxesSubplot):
        is_geoax = True

    # parse styles
    style_dict = _style_dict(style)
    supported_styles = list(_style_dict_raw().keys())
    if (style not in supported_styles) and (style is not None):
        raise ValueError(
            "Given value for `style`(%s) not supported. \
        Currently support [%s]"
            % (style, supported_styles)
        )
    # can I declare these in an automated fashinon?
    bgcolor = style_dict.pop("bgcolor", None)
    fgcolor = style_dict.pop("fgcolor", None)

    fig.patch.set_facecolor(bgcolor)
    # if is_geoax:
    #     ax.background_patch.set_facecolor(bgcolor)
    # else:
    ax.set_facecolor(bgcolor)

    # modify colorbar
    try:
        cb = pp.colorbar
    except AttributeError:
        cb = None

    if cb is not None:
        # set colorbar label plus label color
        cb.set_label(cb.ax.axes.get_ylabel(), color=fgcolor)

        # set colorbar tick color
        cb.ax.yaxis.set_tick_params(color=fgcolor)

        # set colorbar edgecolor
        cb.outline.set_edgecolor(fgcolor)

        # set colorbar ticklabels
        plt.setp(plt.getp(cb.ax.axes, "yticklabels"), color=fgcolor)


def _add_land(ax, style):
    if not isinstance(ax, geoaxes.GeoAxesSubplot):
        raise ValueError("Cannot add land on non-cartopy axes. Got ($s)" % type(ax))
    style_dict = _style_dict(style)
    feature = cfeature.NaturalEarthFeature(
        name="land", category="physical", scale="50m", facecolor=style_dict["landcolor"]
    )
    ax.add_feature(feature)


def _add_coast(ax, style):
    if not isinstance(ax, geoaxes.GeoAxesSubplot):
        raise ValueError("Cannot add land on non-cartopy axes. Got ($s)" % type(ax))
    style_dict = _style_dict(style)
    feature = cfeature.NaturalEarthFeature(
        name="coastline",
        category="physical",
        scale="50m",
        edgecolor=style_dict["coastcolor"],
        facecolor="none",
    )
    ax.add_feature(feature)


### Presets (should proabably put all others into a submodule)
def basic(
    da,
    fig,
    timestamp,
    timestep_value,
    framedim="time",
    plotmethod=None,
    subplot_kw=None,
    **kwargs,
):
    # create axis
    ax = fig.subplots(subplot_kw=subplot_kw)
    pp = _base_plot(
        ax, da, timestamp, timestep_value, framedim, plotmethod=plotmethod, **kwargs
    )
    return ax, pp


def rotating_globe(
    da,
    fig,
    timestamp,
    timestep_value,
    framedim="time",
    plotmethod=None,
    lon_start=-110,
    lon_rotations=0.5,
    lat_start=0,
    lat_rotations=0,
    land=False,
    gridlines=False,
    coastline=True,
    style=None,
    debug=False,
    **kwargs,
):
    # rotate lon_rotations times throughout movie and start at lon_start
    lon = np.linspace(0, 360 * lon_rotations, len(da[framedim])) + lon_start
    # Same for lat
    lat = np.linspace(0, 360 * lat_rotations, len(da[framedim])) + lat_start

    proj = ccrs.NearsidePerspective(
        central_longitude=lon[timestamp], central_latitude=lat[timestamp]
    )

    subplot_kw = dict(projection=proj)

    # mapping style kwargs
    map_style_kwargs = dict(transform=ccrs.PlateCarree())
    kwargs.update(map_style_kwargs)

    # create axis (TODO:this should be handled by the basic preset )
    ax = fig.subplots(subplot_kw=subplot_kw)
    pp = _base_plot(
        ax, da, timestamp, timestep_value, framedim, plotmethod=plotmethod, **kwargs
    )

    _set_style(fig, ax, pp, style=style)

    ax.set_title("")
    ax.set_global()

    # set style (TODO: move this to the basic function including the set style)
    if land:
        _add_land(ax, style)

    if coastline:
        _add_coast(ax, style)

    if gridlines:
        gl = ax.gridlines()
        # Increase gridline res
        gl.n_steps = 500
        # for now fixed locations
        gl.xlocator = mticker.FixedLocator(range(-180, 181, 30))
        gl.ylocator = mticker.FixedLocator(range(-90, 91, 30))
    else:
        gl = None

    return ax, pp

In [8]:
# import xarray as xr
# import dask.bag as db

from dask import delayed, compute
from dask.diagnostics import ProgressBar


# is it a good idea to set these here?
# Needs to be dependent on dpi and videosize
plt.rcParams.update({"font.size": 14})


# Data treatment
def _parse_plot_defaults(da, kwargs):
    if isinstance(da, xr.DataArray):
        data = da
    else:
        raise RuntimeError("input of type (%s) not supported yet." % type(da))

    # check these explicitly to avoid any computation if these are set.
    if "vmin" not in kwargs.keys():
        warnings.warn(
            "No `vmin` provided. Data limits are calculated from input. Depending on the input this can take long. Pass `vmin` to avoid this step",
            UserWarning,
        )
        kwargs["vmin"] = data.min().data

    if "vmax" not in kwargs.keys():
        warnings.warn(
            "No `vmax` provided. Data limits are calculated from input. Depending on the input this can take long. Pass `vmax` to avoid this step",
            UserWarning,
        )
        kwargs["vmax"] = data.max().data

    # There is a bug that prevents this from working...Ill have to fix that upstream.
    # defaults["cbar_kwargs"] = dict(extend="neither")
    # This works for now
    kwargs.setdefault("extend", "neither")

    # if any value is dask.array compute them here.
    for k in ["vmin", "vmax"]:
        if isinstance(kwargs[k], dsa.Array):
            kwargs[k] = kwargs[k].compute()

    return kwargs


def _check_plotfunc_output(func, da, framedim="time", **kwargs):
    timestep = 0
    timestep_value = da[framedim].data[timestep]
    fig = plt.figure()
    oargs = func(da, fig, timestep, timestep_value, framedim, **kwargs)
    # I just want the number of output args, delete plot
    plt.close(fig)
    if oargs is None:
        return 0
    else:
        return len(oargs)


def _check_ffmpeg_version():
    p = Popen("ffmpeg -version", stdout=PIPE, shell=True)
    (output, err) = p.communicate()
    p_status = p.wait()
    # Parse version
    if p_status != 0:
        print("No ffmpeg found")
        return None
    else:
        # parse version number
        try:
            found = (
                re.search("ffmpeg version (.+?) Copyright", str(output))
                .group(1)
                .replace(" ", "")
            )
            return found
        except AttributeError:
            # ffmpeg version, Copyright not found in the original string
            found = None
    return found


def _execute_command(
    command, verbose=False, error=True, log_file="output.log", max_lines=10
):
    with open(log_file, "w") as f:
        p = Popen(
            command,
            stdout=PIPE,
            stderr=STDOUT,
            shell=True,
            bufsize=1,
            universal_newlines=True,
        )
        line_count = 0

        for line in iter(p.stdout.readline, ""):
            f.write(line)  # Write to log file

            if verbose and line_count < max_lines:
                sys.stdout.write(line)  # Display line in console
                sys.stdout.flush()
                line_count += 1

        p.stdout.close()
        p.wait()

    # Inform if output was truncated
    if verbose and line_count >= max_lines:
        print(f"\n...Output truncated. Full log saved to {log_file}")

    if error and p.returncode != 0:
        raise RuntimeError(
            f"Command '{command}' failed with return code {p.returncode}"
        )

    return p


def _check_ffmpeg_execute(command, verbose=False):
    if _check_ffmpeg_version() is None:
        raise RuntimeError(
            "Could not find an ffmpeg version on the system. \
        Please install ffmpeg with e.g. `conda install -c conda-forge ffmpeg`"
        )
    else:
        try:
            p = _execute_command(command, verbose=verbose)
            return p
        except RuntimeError:
            raise RuntimeError(
                "Something has gone wrong. Use `verbose=True` to check if ffmpeg displays a problem"
            )


def convert_gif(
    mpath,
    gpath="movie.gif",
    gif_palette=False,
    resolution=[480, 320],
    verbose=False,
    remove_movie=True,
    gif_framerate=5,
):
    if gif_palette:
        palette_filter = (
            '-filter_complex "[0:v] split [a][b];[a] palettegen [p];[b][p] paletteuse"'
        )
    else:
        palette_filter = ""

    command = "ffmpeg -y -i %s %s -r %i -s %ix%i %s" % (
        mpath,
        palette_filter,
        gif_framerate,
        resolution[0],
        resolution[1],
        gpath,
    )
    p = _check_ffmpeg_execute(command, verbose=verbose)

    print("GIF created at %s" % (gpath))
    if remove_movie:
        if os.path.exists(mpath):
            os.remove(mpath)
    return p


def _combine_ffmpeg_command(
    sourcefolder, moviename, framerate, frame_pattern, ffmpeg_options
):
    # we need `-y` because i can not properly diagnose the errors here...
    command = 'ffmpeg -r %i -i "%s" -y %s -r %i "%s"' % (
        framerate,
        os.path.join(sourcefolder, frame_pattern),
        ffmpeg_options,
        framerate,
        os.path.join(sourcefolder, moviename),
    )
    return command


def write_movie(
    sourcefolder,
    moviename,
    frame_pattern="frame_%05d.png",
    remove_frames=True,
    verbose=False,
    ffmpeg_options="-c:v libvpx-vp9 -b:v 2M -f mp4",
    framerate=20,
):
    command = _combine_ffmpeg_command(
        sourcefolder, moviename, framerate, frame_pattern, ffmpeg_options
    )
    p = _check_ffmpeg_execute(command, verbose=verbose)

    print("Movie created at %s" % (moviename))
    if remove_frames:
        rem_name = frame_pattern.replace("%05d", "*")
        for f in glob.glob(os.path.join(sourcefolder, rem_name)):
            if os.path.exists(f):
                os.remove(f)
    return p


def frame_save(fig, frame, odir=None, frame_pattern="frame_%05d.png", dpi=100):
    fig.savefig(
        os.path.join(odir, frame_pattern % (frame)),
        dpi=dpi,
        facecolor=fig.get_facecolor(),
        transparent=True,
    )
    # I am trying everything to *wipe* this figure, hoping that it could
    # help with the dask glitches I experienced earlier.
    # TBD if this is all needed...how this might affect performance.
    plt.close(fig)
    del fig
    gc.collect(2)


class Movie:
    def __init__(
        self,
        da,
        plotfunc=None,
        framedim="time",
        pixelwidth=1920,
        pixelheight=1080,
        dpi=200,
        frame_pattern="frame_%05d.png",
        fieldname=None,
        input_check=True,
        **kwargs,
    ):
        self.pixelwidth = pixelwidth
        self.pixelheight = pixelheight
        self.dpi = dpi
        self.width = self.pixelwidth / self.dpi
        self.height = self.pixelheight / self.dpi
        self.frame_pattern = frame_pattern
        self.data = da
        self.framedim = framedim
        if plotfunc is None:
            self.plotfunc = basic
        else:
            self.plotfunc = plotfunc
        # set sensible defaults
        self.raw_kwargs = kwargs

        # Check input

        # optional checks (these might need to be deactivated when using custom
        # plot functions.)
        if input_check:
            if isinstance(self.data, xr.Dataset):
                raise ValueError(
                    "xmovie presets do not yet support the input of xr.Datasets. \
                In order to use datasets as inputs, set `input_check` to False. \
                Note that this requires you to manually set colorlimits etc."
                )

            # Set defaults
            self.kwargs = _parse_plot_defaults(self.data, self.raw_kwargs)
        else:
            self.kwargs = self.raw_kwargs

        # Mandatory checks
        # Check if `framedim` exists.
        if self.framedim not in list(self.data.dims):
            raise ValueError("Framedim (%s) not found in input data" % self.framedim)
        # Check the output of plotfunc
        self.plotfunc_n_outargs = _check_plotfunc_output(
            self.plotfunc, self.data, self.framedim, **self.kwargs
        )

    def render_frame(self, timestep, timestep_value):
        """renders complete figure (frame) for given timestep.

        Parameters
        ----------
        timestep : type
            Description of parameter `timestep`.

        Returns
        -------
        type
            Description of returned object.

        """
        fig = plt.figure(figsize=[self.width, self.height])
        # create_frame(self.pixelwidth, self.pixelheight, self.dpi)
        # produce dummy output for ax and pp if the plotfunc does not provide them
        if self.plotfunc_n_outargs == 2:
            # this should be the case for all presets provided by xmovie
            ax, pp = self.plotfunc(
                self.data, fig, timestep, timestep_value, self.framedim, **self.kwargs
            )
        else:
            warnings.warn(
                "The provided `plotfunc` does not provide the expected number of output arguments.\
            Expected a function `ax,pp =plotfunc(...)` but got %i output arguments. Inserting dummy values. This should not affect output. ",
                UserWarning,
            )
            _ = self.plotfunc(
                self.data, fig, timestep, timestep_value, self.framedim, **self.kwargs
            )
            ax, pp = None, None
        return fig, ax, pp

    def save_frames(self, odir, progress=False):
        """Save movie frames as picture files.

        Parameters
        ----------
        odir : path
            path to output directory
        progress : type
            Show progress bar. Requires

        """
        # create range of frames
        timesteps = self.data[self.framedim].data
        frame_range = range(len(timesteps))
        if tqdm_avail and progress:
            frame_range = tqdm(frame_range)
        elif ~tqdm_avail and progress:
            warnings.warn("Cant show progess bar at this point. Install tqdm")

        for fi in frame_range:
            fig, ax, pp = self.render_frame(fi, timesteps[fi])
            frame_save(
                fig, fi, odir=odir, frame_pattern=self.frame_pattern, dpi=self.dpi
            )

    # Needs more testing! Slower than the sequential counterpart
    def save_frames_parallel(self, odir, batch_size=10, progress=False):
        """Save frames in parallel batches to reduce I/O overhead."""
        timesteps = self.data[self.framedim].data
        frame_range = range(len(timesteps))

        for i in tqdm(range(0, len(frame_range), batch_size)):
            batch = frame_range[i : i + batch_size]
            tasks = []
            for fi in batch:
                delayed_frame = delayed(self.render_frame)(fi, timesteps[fi])
                tasks.append(
                    delayed(frame_save)(
                        delayed_frame[0],
                        fi,
                        odir=odir,
                        frame_pattern=self.frame_pattern,
                        dpi=self.dpi,
                    )
                )
            compute(*tasks)

    def save(
        self,
        filename,
        remove_frames=True,
        remove_movie=True,
        progress=False,
        verbose=False,
        overwrite_existing=False,
        framerate=15,
        ffmpeg_options="-c:v mjpeg -q:v 2 -pix_fmt yuvj420p",
        gif_palette=False,
        gif_resolution_factor=0.5,
        gif_framerate=10,
        parallel=False,
        batch_size=10,
    ):
        """Save out animation from Movie object.

        Parameters
        ----------
        filename : str
            Pathname to final movie/animation. Output is dependent on filetype:
            Creates movie for `*.mp4` and gif for `*.gif`
        remove_frames : Bool
            Optional removal of frame pictures (the default is True; False will
            leave all picture files in folder).
        remove_movie : Bool
            As `remove_frames` but for movie file. Only applies when filename
            is given as `.gif` (the default is True).
        progress : Bool
            Experimental switch to show progress output. This will be refined
            in future version (the default is False).
        verbose : Bool
            Experimental switch to show output of ffmpeg commands. Useful for
            debugging but can quickly flood your notebook
            (the default is False).
        overwrite_existing : Bool
            Set to overwrite existing files with `filename`
            (the default is False).
        framerate : int
            Frames per second for the output movie file. Only relevant for '.mp4' files.
            (the default is 15)
        ffmpeg_options: str
            Encoding options to pass to ffmpeg call.
            Defaults to : `"-c:v libx264 -preset veryslow -crf 10 -pix_fmt yuv420p"`
        gif_palette : Bool
            Use a gif colorpalette to improve quality. Can lead to artifacts
            in very contrasty situations (the default is False).
        gif_resolution_factor : float
            Factor used to reduce gif resolution compared to movie.
            Use 1.0 to put out the same resolutions for both products.
            (the default is 0.5).
        gif_framerate : int
            As `framerate` but for the gif output file. Only relevant to `.gif` files.
            (the default is 10)
        """

        # parse out directory and filename
        dirname = os.path.dirname(filename)
        filename = os.path.basename(filename)

        # detect gif filename

        isgif = ".gif" in filename
        if isgif:
            giffile = filename
            moviefile = filename.replace("gif", "mp4")
            gpath = os.path.join(dirname, giffile)
        else:
            moviefile = filename

        mpath = os.path.join(dirname, moviefile)

        # check existing files
        if os.path.exists(mpath):
            if not overwrite_existing:
                raise RuntimeError(
                    "File `%s` already exists. Set `overwrite_existing` to True to overwrite."
                    % (mpath)
                )
        if isgif:
            if os.path.exists(gpath):
                if not overwrite_existing:
                    raise RuntimeError(
                        "File `%s` already exists. Set `overwrite_existing` to True to overwrite."
                        % (gpath)
                    )

        # print frames
        if parallel:
            self.save_frames_parallel(dirname, batch_size=batch_size, progress=progress)
        else:
            self.save_frames(dirname, progress=progress)

        # Create movie
        write_movie(
            dirname,
            moviefile,
            frame_pattern=self.frame_pattern,
            remove_frames=remove_frames,
            verbose=verbose,
            framerate=framerate,
            ffmpeg_options=ffmpeg_options,
        )

        # Create gif
        if isgif:
            # if ppath:
            #     create_gif_palette(mpath, ppath=ppath, verbose=verbose)
            convert_gif(
                mpath,
                gpath=gpath,
                resolution=[480, 320],
                gif_palette=gif_palette,
                verbose=verbose,
                remove_movie=remove_movie,
                gif_framerate=gif_framerate,
            )

### Basic Movies

In [17]:
ds = (ds_prediction.thetao - ds_groundtruth.thetao).isel(lev=0).compute()
ds = ds.isel(time=slice(0, 50))

/pscratch/sd/s/suryad/conda_envs/emulator_scratch/lib/python3.10/site-packages/dask/array/core.py:4836: PerformanceWarning: Increasing number of chunks by factor of 80
  result = blockwise(


In [14]:
import cmocean as cm

mov = Movie(
    ds,
    style="dark",
    plotfunc=rotating_globe,
    land=False,
    coastline=False,
    lon_rotations=1,
    lon_start=120,
    lat_start=25,
    cmap=cm.cm.balance,
    vmin=-2,
    vmax=2,
)
mov.save(
    prefix + "_movie_bias_rotating_om4.mp4", progress=True, overwrite_existing=True
)

AttributeError: module 'dask' has no attribute 'Array'

In [19]:
import cmocean as cm

mov = Movie(ds, cmap=cm.cm.balance, vmin=-2, vmax=2)
mov.save(
    prefix + "_movie_bias_om4.mp4",
    progress=True,
    overwrite_existing=True,
    parallel=True,
)

  0%|          | 0/5 [00:00<?, ?it/s]

Movie created at CM4_movie_bias_om4.mp4


In [65]:
import cmocean as cm

mov = Movie(
    ds_prediction.thetao.isel(lev=0),
    style="dark",
    plotfunc=rotating_globe,
    land=False,
    coastline=False,
    lon_rotations=1,
    lon_start=120,
    lat_start=25,
    cmap=cm.cm.balance,
)
mov.save(
    prefix + "_movie_pred_rotating_om4.mp4", progress=True, overwrite_existing=True
)

### ENSO

In [9]:
data = ds_groundtruth

In [10]:
clim = data["thetao"].sel(lev=slice(0, 500)).groupby("time.dayofyear").mean().compute()
data_surface = data.sel(lev=slice(0, 500))
ds_prediction_surface = ds_prediction.sel(lev=slice(0, 500))
clim_pred = ds_prediction_surface["thetao"].groupby("time.dayofyear").mean().compute()

In [11]:
def NinoIndexComputeClim(T, area, dt=5, window=150):
    T = T.load()
    T_clim = T.copy()
    T_clim = T_clim.sel(x=slice(118, 260), y=slice(-5, 5))
    area = area.sel(x=slice(118, 260), y=slice(-5, 5)).load()
    clim = T_clim.groupby("time.dayofyear").mean("time").compute()
    window = int(window / dt)
    for i, t in enumerate(T_clim.time.values):
        day = int(t.dayofyr)
        T_clim[i] = (T[i] - clim.sel(dayofyear=day)).data

    T_clim = T_clim.rolling(time=window).mean()
    # T_clim = (T_clim*area).sum(["x","y"])/area.sum(["x","y"])
    # T_clim = T_clim.weighted(area).mean(["x", "y"])
    T_clim = T_clim.weighted(area).mean(["y"])

    return T_clim[window:]

In [12]:
nino_true_compute_clim = NinoIndexComputeClim(data_surface["thetao"], data["areacello"])
nino_true_compute_clim = nino_true_compute_clim.rename("Anomaly")
nino_true_compute_clim = nino_true_compute_clim.assign_attrs(units=r"$\degree C$")
nino_true_compute_clim["x"] = nino_true_compute_clim["x"].assign_attrs(
    units=r"${^o}$", long_name="longitude"
)
nino_true_compute_clim["lev"] = nino_true_compute_clim["lev"].assign_attrs(
    units=r"$m$", long_name="depth"
)

nino_pred_compute_clim = NinoIndexComputeClim(
    ds_prediction_surface["thetao"], ds_prediction_surface["areacello"]
)
nino_pred_compute_clim = nino_pred_compute_clim.rename("Anomaly")
nino_pred_compute_clim = nino_pred_compute_clim.assign_attrs(units=r"$\degree C$")
nino_pred_compute_clim["x"] = nino_pred_compute_clim["x"].assign_attrs(
    units=r"${^o}$", long_name="longitude"
)
nino_pred_compute_clim["lev"] = nino_pred_compute_clim["lev"].assign_attrs(
    units=r"$m$", long_name="depth"
)

#### ENSO Bias

In [32]:
ds = (nino_pred_compute_clim - nino_true_compute_clim).compute()
ds = ds.isel(time=slice(0, 50))
ds

<xarray.DataArray 'Anomaly' (time: 50, x: 142, lev: 9)> Size: 511kB
array([[[-0.0303532 , -0.02084642, -0.08746928, ..., -0.34787919,
          0.05982494,  0.06688325],
        [-0.07073615, -0.06388551, -0.09664036, ..., -0.35685473,
          0.0698171 ,  0.0720442 ],
        [-0.01173211,  0.03013121, -0.00735835, ..., -0.3376964 ,
         -0.01461165,  0.04210373],
        ...,
        [ 0.08817904,  0.26037458,  0.39422349, ...,  0.11840884,
          0.06208936,  0.03986695],
        [ 0.09879181,  0.30519812,  0.47694337, ...,  0.11840296,
          0.06179618,  0.0398139 ],
        [ 0.11650619,  0.35458084,  0.56409704, ...,  0.11938508,
          0.06319138,  0.04189197]],

       [[-0.03618929, -0.02917074, -0.09620325, ..., -0.38134238,
          0.04687224,  0.06379021],
        [-0.0732348 , -0.06909055, -0.10903023, ..., -0.39145657,
          0.05743632,  0.06948551],
        [-0.0144715 ,  0.02140466, -0.01302384, ..., -0.36320073,
         -0.020634  ,  0.04400865],
...
        [-0.11320295, -0.11004686, -0.19092452, ...,  0.08121379,
          0.01146064, -0.03960478],
        [-0.09687855, -0.08437748, -0.17465066, ...,  0.05506643,
         -0.00327604, -0.03721296],
        [-0.0874443 , -0.06827087, -0.1788897 , ...,  0.0350465 ,
         -0.01853634, -0.03296809]],

       [[-0.16634969, -0.15768361, -0.11183253, ..., -0.12231527,
          0.16090866,  0.13774566],
        [-0.13714805, -0.14174241, -0.09056735, ..., -0.11607175,
          0.15535962,  0.11910829],
        [-0.07441268, -0.01024726, -0.07472898, ..., -0.12950876,
          0.18323346,  0.16686984],
        ...,
        [-0.07344313, -0.08356041, -0.17513198, ...,  0.07252574,
          0.0061449 , -0.04320297],
        [-0.06107865, -0.06088451, -0.16049203, ...,  0.0471386 ,
         -0.00723268, -0.04019015],
        [-0.05690512, -0.04963102, -0.16799564, ...,  0.02918106,
         -0.02038721, -0.03474345]]], shape=(50, 142, 9))
Coordinates:
    dz       (lev) int64 72B 5 10 15 20 30 50 70 100 150
  * time     (time) object 400B 0342-05-26 00:00:00 ... 0343-01-26 00:00:00
  * x        (x) float64 1kB 118.5 119.5 120.5 121.5 ... 256.5 257.5 258.5 259.5
  * lev      (lev) float64 72B 2.5 10.0 22.5 40.0 65.0 105.0 165.0 250.0 375.0

In [13]:
def enso(da, fig, timestamp, timestamp_value, framedim="time", title="", **kwargs):
    ax = fig.subplots()
    data = da.isel({framedim: timestamp})
    pp = data.plot.pcolormesh(ax=ax, **kwargs)

    ax.set_title(title)
    ax.set_xlabel("")
    ax.invert_yaxis()

    return ax, pp

In [14]:
import cmocean as cm

mov = Movie(
    ds, plotfunc=enso, y="lev", cmap=cm.cm.balance, vmin=-2, vmax=2, title="ENSO Bias"
)
mov.save(prefix + "_movie_enso_bias_om4.mp4", progress=True, overwrite_existing=True)

  0%|          | 0/50 [00:00<?, ?it/s]

Movie created at CM4_movie_enso_bias_om4.mp4


#### ENSO Both 

In [13]:
def enso2(da, fig, timestamp, timestamp_val, framedim="time", title="", **kwargs):
    da_pred = da[0]
    da_gt = da[1]
    # Plot both one on top of the other
    ax = fig.subplots(2, 1, sharex=True)
    data_pred = da_pred.isel({framedim: timestamp})
    data_gt = da_gt.isel({framedim: timestamp})
    pp_pred = data_pred.plot.pcolormesh(ax=ax[0], **kwargs)
    pp_gt = data_gt.plot.pcolormesh(ax=ax[1], **kwargs)

    ax[0].set_title(title)
    ax[0].set_xlabel("")
    ax[0].invert_yaxis()
    ax[1].invert_yaxis()
    ax[1].set_title(dataset_name)
    ax[0].set_title("Samudra")

    # Text for current date based on timestamp
    ax[0].text(
        1.0,
        1.2,
        f"{timestamp_val.year}-{timestamp_val.month}-{timestamp_val.day}",
        horizontalalignment="center",
        verticalalignment="center",
        transform=ax[0].transAxes,
        fontsize=14,
        fontweight="bold",
    )

    return ax, pp_pred

In [14]:
# combine the two datasets into a single xarray new dimension
da = xr.concat([nino_pred_compute_clim, nino_true_compute_clim], dim="dummy")

mov = Movie(
    da, plotfunc=enso2, y="lev", cmap=cm.cm.balance, vmin=-2, vmax=2, title="ENSO"
)
mov.save(prefix + "_movie_enso_both.mp4", progress=True, overwrite_existing=True)

  0%|          | 0/570 [00:00<?, ?it/s]

Movie created at CM4_movie_enso_both.mp4


### Global Map

In [43]:
import matplotlib.pyplot as plt
import numpy as np
import os
import cmocean as cm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.ticker import FixedLocator


def global_surface_map(da, fig, timestamp, timestamp_val, framedim="time", **kwargs):
    var_name = kwargs["var_name"]

    da_pred = da.isel(dummy=0)
    da_gt = da.isel(dummy=1)
    data_pred = da_pred.isel({framedim: timestamp})
    data_gt = da_gt.isel({framedim: timestamp})

    plt.clf()
    plt.rcParams.update({"font.size": 14})

    # Define colormap
    new_cmap = cm.cm.thermal if var_name == "thetao" else cm.cm.haline
    new_cmap.set_bad(color=(0.7, 0.7, 0.7, 0))

    # Set common color range for the colorbar
    vmin, vmax = {
        "thetao": (0, 30),
        "so": (30, 40),
        "uo": (-2, 2),
        "vo": (-2, 2),
        "zos": (-1, 1),
    }[var_name]

    # Create figure with appropriate layout
    ax = fig.subplots(
        1,
        2,
        subplot_kw={"projection": ccrs.PlateCarree()},
        gridspec_kw={"wspace": 0.02, "hspace": 0.05},
    )
    ax = np.array(ax)  # Ensure ax is an array for easy indexing

    # Plot Ground Truth (GT)
    im = data_gt.plot(
        ax=ax[0],
        cmap=new_cmap,
        vmin=vmin,
        vmax=vmax,
        add_colorbar=False,
    )
    # ax[0].add_feature(cfeature.COASTLINE, edgecolor="black")
    ax[0].set_title(f"{dataset_name}", fontsize=14)
    gl = ax[0].gridlines(draw_labels=True, color="0.4", linestyle="--", alpha=0)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 14}
    gl.ylabel_style = {"size": 14}
    gl.xlocator = FixedLocator([-120, -60, 0, 60, 120])

    # Plot Predictions
    im = data_pred.plot(
        ax=ax[1],
        cmap=new_cmap,
        vmin=vmin,
        vmax=vmax,
        add_colorbar=False,
    )
    # ax[1].add_feature(cfeature.COASTLINE, edgecolor="black")
    ax[1].set_title("Samudra", fontsize=14)
    gl = ax[1].gridlines(draw_labels=True, color="0.4", linestyle="--", alpha=0)
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = False
    gl.xlabel_style = {"size": 14}
    gl.ylabel_style = {"size": 14}
    gl.xlocator = FixedLocator([-120, -60, 0, 60, 120])
    ax[1].set_yticks([])
    ax[1].set_ylabel("")

    # Add colorbar for both plots
    cbar = fig.colorbar(im, ax=ax[:], orientation="vertical", fraction=0.02, pad=0.02)
    cbar.set_label(var_list[var_name])

    # Add timestamp text
    ax[0].text(
        1.0,
        1.2,
        f"{timestamp_val.year}-{timestamp_val.month:02d}-{timestamp_val.day:02d}",
        horizontalalignment="center",
        verticalalignment="center",
        transform=ax[0].transAxes,
        fontsize=14,
        fontweight="bold",
    )

    return ax, im

In [45]:
# combine the two datasets into a single xarray new dimension
var = "thetao"
data_pred = ds_prediction[var].isel(lev=0).compute()
data_gt = ds_groundtruth[var].isel(lev=0).compute()

da = xr.concat([data_pred, data_gt], dim="dummy")

mov = Movie(da, plotfunc=global_surface_map, var_name=var, input_check=False)
mov.save(
    prefix + f"_movie_{var}_surfacemap_both_{dataset_name}.mp4",
    progress=True,
    overwrite_existing=True,
)

  0%|          | 0/600 [00:00<?, ?it/s]

Movie created at CM4_movie_thetao_surfacemap_both_CM4.mp4


### Global Profile

In [37]:
from xarrayutils.plotting import linear_piecewise_scale
import matplotlib.pyplot as plt
import numpy as np
import os
import cmocean as cm


def global_profile(da, fig, timestamp, timestamp_val, framedim="time", **kwargs):
    da_pred = da.isel(dummy=0)
    da_gt = da.isel(dummy=1)
    data_pred = da_pred.isel({framedim: timestamp})
    data_gt = da_gt.isel({framedim: timestamp})
    var_name = kwargs["var_name"]

    plt.clf()
    plt.rcParams.update({"font.size": 14})

    # Define colormap
    new_cmap = cm.cm.thermal if var_name == "thetao" else cm.cm.haline
    new_cmap.set_bad("grey", 0.6)

    # Create figure with appropriate layout
    ax = fig.subplots(
        2,
        1,
        gridspec_kw={
            "width_ratios": [1] * 1,
            "height_ratios": [1] * 2,
            "wspace": 0.02,
            "hspace": 0.5,
        },
    )
    ax = np.array(ax)  # Ensure ax is an array for easy indexing

    # Set common color range for the colorbar
    vmin, vmax = {
        "thetao": (0, 30),
        "so": (30, 40),
        "uo": (-2, 2),
        "vo": (-2, 2),
        "zos": (-1, 1),
    }[var_name]

    # Plot GT (original data)
    im = data_gt.plot(
        ax=ax[0], cmap=new_cmap, vmin=vmin, vmax=vmax, add_colorbar=False, y="lev"
    )
    ax[0].invert_yaxis()
    ax[0].set_title(dataset_name, fontsize=14)
    ax[0].set_xticks([])
    ax[0].set_xlabel("")

    # Plot predictions from other models
    im = data_pred.plot(
        ax=ax[1], cmap=new_cmap, vmin=vmin, vmax=vmax, add_colorbar=False, y="lev"
    )
    ax[1].invert_yaxis()
    ax[1].set_title("Samudra", fontsize=14)

    # Add shared colorbar for each row
    cbar = fig.colorbar(im, ax=ax[:], orientation="vertical", fraction=0.02, pad=0.02)
    cbar.set_label(var_list[var_name])

    # Text for current date based on timestamp
    ax[0].text(
        1.0,
        1.2,
        f"{timestamp_val.year}-{timestamp_val.month}-{timestamp_val.day}",
        horizontalalignment="center",
        verticalalignment="center",
        transform=ax[0].transAxes,
        fontsize=14,
        fontweight="bold",
    )

    return ax, im

In [34]:
# combine the two datasets into a single xarray new dimension
var = "thetao"

# Prediction
da_pred = ds_prediction[var].sel(lev=slice(0, 500))
section_mask = np.isnan(da_pred).all("x").isel(time=0)
da_pred_int_x = da_pred.weighted(da_pred["areacello"]).mean(["x"])
var_pred = da_pred_int_x.where(~section_mask)
var_pred = var_pred.rename(var_list[var]).assign_attrs(units=var_list[var])
var_pred["y"] = var_pred.y.assign_attrs(long_name="latitude", units=r"$\degree$")
var_pred["lev"] = var_pred.lev.assign_attrs(long_name="depth", units="m")
var_pred = var_pred.compute()

# GT
da_gt = ds_groundtruth[var].sel(lev=slice(0, 500))
section_mask = np.isnan(da_gt).all("x").isel(time=0)
da_gt_int_x = da_gt.weighted(da_gt["areacello"]).mean(["x"])
var_gt = da_gt_int_x.where(~section_mask)
var_gt = var_gt.rename(var_list[var]).assign_attrs(units=var_list[var])
var_gt["y"] = var_gt.y.assign_attrs(long_name="latitude", units=r"$\degree$")
var_gt["lev"] = var_gt.lev.assign_attrs(long_name="depth", units="m")
var_gt = var_gt.compute()

da = xr.concat([var_pred, var_gt], dim="dummy")

In [38]:
mov = Movie(da, plotfunc=global_profile, var_name=var, input_check=False)
mov.save(
    prefix + f"_movie_{var}_profile_both_{dataset_name}.mp4",
    progress=True,
    overwrite_existing=True,
)

  0%|          | 0/600 [00:00<?, ?it/s]

Movie created at CM4_movie_thetao_profile_both_CM4.mp4
